# ⚡ Introduction to CUDA with nvcc4jupyter

This notebook demonstrates how to load the extension, set global compiler flags for the RTX 4060, and run/profile native CUDA C++ code.

In [ ]:
# Load the CUDA compiler extension
%load_ext nvcc4jupyter

In [ ]:
# Apply Ada Lovelace architecture and max optimization flag for the entire notebook
import nvcc4jupyter
nvcc4jupyter.set_defaults(compiler_args="-gencode arch=compute_89,code=sm_89 -O3")

In [ ]:
%%cuda
#include <iostream>

__global__ void mathKernel(float *out) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    out[idx] = (idx * 3.14f) + 1.0f;
}

int main() {
    int threads = 256;
    float *d_out;
    
    cudaMalloc(&d_out, threads * sizeof(float));
    mathKernel<<<1, threads>>>(d_out);
    cudaDeviceSynchronize();
    
    std::cout << "Kernel executed successfully with injected flags!" << std::endl;
    
    cudaFree(d_out);
    return 0;
}

In [ ]:
%%cuda --profile --profiler-args "--section SpeedOfLight"
#include <iostream>
#include <math.h>

__global__ void heavyMathKernel(float *out) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    float val = (float)idx;
    
    for(int i = 0; i < 1000; i++) {
        val = sinf(val) * cosf(val);
    }
    
    out[idx] = val;
}

int main() {
    int blocks = 1024;
    int threads = 256;
    float *d_out;
    
    cudaMalloc(&d_out, blocks * threads * sizeof(float));
    heavyMathKernel<<<blocks, threads>>>(d_out);
    cudaDeviceSynchronize();
    
    std::cout << "Profiling complete. Check the output above for the Speed Of Light metrics." << std::endl;

    cudaFree(d_out);
    return 0;
}